<a href="https://colab.research.google.com/github/AndrijaM06/car-price-prediction/blob/main/03_feature_engineering.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Inženjering karakteristika: cars.csv

Karakteristike koje pravimo:
- `car_age` - starost automobila u godinama
- `mileage_per_year` - prosečna kilometraža po godini starosti
- `engine_volume_liters` - zapremina motora u litrima umesto cm³
- `is_newer_car` - indikator da li je automobil novijeg godišta (2010+)
- `is_high_mileage` - indikator da li automobil ima veoma veliku kilometražu


## Učitavanje očišćenog skupa podataka

In [12]:
from pathlib import Path
import pandas as pd

url = "https://raw.githubusercontent.com/AndrijaM06/car-price-prediction/main/data/cars_cleaned.csv"
df = pd.read_csv(url)
df.shape

(55585, 12)

In [13]:
df.head()

,make,model,price_usd,year,condition,mileage_km,fuel_type,volume_cm3,color,transmission,drive_unit,segment
0,mazda,2,5500,2008,with mileage,162000.0,petrol,1500.0,burgundy,mechanics,front-wheel drive,b
1,mazda,2,5350,2009,with mileage,120000.0,petrol,1300.0,black,mechanics,front-wheel drive,b
2,mazda,2,7000,2009,with mileage,61000.0,petrol,1500.0,silver,auto,front-wheel drive,b
3,mazda,2,3300,2003,with mileage,265000.0,diesel,1400.0,white,mechanics,front-wheel drive,b
4,mazda,2,5200,2008,with mileage,97183.0,diesel,1400.0,gray,mechanics,front-wheel drive,b


## 1. `car_age` - starost automobila

Umesto da model direktno koristi `year` (npr. 2008), korisnije mu je da zna
koliko je automobil star, jer je odnos starosti i cene intuitivniji (stariji
automobil → obično jeftiniji).


In [14]:
REFERENCE_YEAR = 2026

df["car_age"] = REFERENCE_YEAR - df["year"]
df[["year", "car_age"]].describe()

,year,car_age
count,55585.000000,55585.000000
mean,2003.617325,22.382675
std,7.863027,7.863027
min,1970.000000,7.000000
25%,1998.000000,16.000000
50%,2004.000000,22.000000
75%,2010.000000,28.000000
max,2019.000000,56.000000


## 2. `mileage_per_year` - prosečna kilometraža po godini

Ovo je informativnija karakteristika od same kilometraže, jer 150.000 km na
5-godišnjem automobilu je mnogo intenzivnije korišćenje nego 150.000 km na
20-godišnjem automobilu.

Koristimo `clip(lower=1)` da izbegnemo deljenje sa nulom za automobile
proizvedene u referentnoj godini (car_age = 0).

In [15]:
safe_age = df["car_age"].clip(lower=1)
df["mileage_per_year"] = df["mileage_km"] / safe_age

df[["mileage_km", "car_age", "mileage_per_year"]].head()

,mileage_km,car_age,mileage_per_year
0,162000.0,18,9000.000000
1,120000.0,17,7058.823529
2,61000.0,17,3588.235294
3,265000.0,23,11521.739130
4,97183.0,18,5399.055556


In [16]:
df["mileage_per_year"].describe()

,mileage_per_year
count,55585.000000
mean,10235.949422
std,4740.134770
min,0.000000
25%,7714.285714
50%,10500.000000
75%,13157.894737
max,50000.000000


## 3. `engine_volume_liters` - zapremina motora u litrima

Zapremina u litrima (npr. 1.6, 2.0) je čitljivija karakteristika od
zapremine u cm³ (1600, 2000), iako nosi identičnu informaciju. Ovo je više
kozmetička nego suštinska promena, ali pomaže pri tumačenju rezultata.

In [17]:
df["engine_volume_liters"] = df["volume_cm3"] / 1000
df[["volume_cm3", "engine_volume_liters"]].head()

,volume_cm3,engine_volume_liters
0,1500.0,1.5
1,1300.0,1.3
2,1500.0,1.5
3,1400.0,1.4
4,1400.0,1.4


## 4. `is_newer_car` - indikator novijeg godišta

Binarni indikator da li je automobil proizveden 2015. godine ili kasnije.
Ovo modelu daje jednostavan signal "novije/starije" pored kontinuirane
vrednosti `car_age`.

In [18]:
NEWER_CAR_YEAR_THRESHOLD = 2015

df["is_newer_car"] = (df["year"] >= NEWER_CAR_YEAR_THRESHOLD).astype(int)
df["is_newer_car"].value_counts()

,count
is_newer_car,
0,50774
1,4811


## 5. `is_high_mileage` - indikator visoke kilometraže

Prag od 300.000 km smo izabrali na osnovu EDA analize - to je otprilike
75-ti percentil očišćenog skupa podataka, što znači da ova karakteristika
izdvaja gornju četvrtinu automobila po kilometraži.

In [19]:
HIGH_MILEAGE_THRESHOLD_KM = 300_000

df["is_high_mileage"] = (df["mileage_km"] > HIGH_MILEAGE_THRESHOLD_KM).astype(int)
df["is_high_mileage"].value_counts()

,count
is_high_mileage,
0,41134
1,14451


## Provera korelacije novih karakteristika sa cenom

Pre nego što pređemo dalje, korisno je da vidimo da li nove karakteristike
uopšte imaju vezu sa ciljnom promenljivom `price_usd`.

In [20]:
numeric_cols_to_check = [
    "price_usd", "year", "mileage_km", "volume_cm3",
    "car_age", "mileage_per_year", "engine_volume_liters",
]

df[numeric_cols_to_check].corr()["price_usd"].sort_values(ascending=False)

,price_usd
price_usd,1.000000
year,0.621838
engine_volume_liters,0.372690
volume_cm3,0.372690
mileage_per_year,-0.020878
mileage_km,-0.340579
car_age,-0.621838


**Zaključak:** `car_age` ima jaku negativnu korelaciju sa cenom (stariji
automobil → niža cena), što je očekivano i potvrđuje da je ova karakteristika
korisna. `mileage_per_year` takođe pokazuje negativnu korelaciju.

## Zaključak

Od očišćenog skupa podataka (12 kolona) dobili smo skup sa **17 kolona** -
dodali smo 5 novih karakteristika koje bolje opisuju automobil za potrebe
predviđanja cene.

Rezultat je sačuvan u `data/cars_features.csv` (kroz skript) i spreman je za
sledeći korak: **pretprocesiranje podataka**, gde ćemo kolone pripremiti u
oblik koji regresioni model može direktno da koristi.